# Proteome exploration with **sparse SAE feature** embeddings

A companion to `explore_proteome.ipynb`. Everything is the same — one scanpy graph driving a UMAP layout and Leiden clusters, with per-cluster SAE-feature enrichment and ESM Atlas annotation — **except the basis for the KNN graph**. Here the neighbors graph is built directly on the **sparse top-K SAE feature activations** (`adata.X`, proteins × SAE features), not the dense ESMC embeddings. Clustering on the interpretable feature space means each cluster is, by construction, a set of proteins sharing the same active SAE features.

**Prerequisites:** run the embedding step *with SAE enabled* first, and `pip install -e "..[cluster]"` (scanpy, leidenalg, igraph). Needs `BASEROW_TOKEN` / `BIOHUB_API_TOKEN` in the env.

In [ ]:
from och_annotate.config import load_config
from och_annotate.analysis import load_embeddings

cfg = load_config("../config/octopus_chierchiae.yaml")
df = load_embeddings(cfg, prefer_cache=True)   # backfills Baserow metadata cols not in cache
print(f"Loaded {len(df)} embeddings; vector dim = {len(df['embedding'].iloc[0])}")
df.head()

## SAE feature matrix → local store (outside Baserow)

The SAE feature vector per protein is persisted **outside Baserow** as one sparse `float32` `.npz` (`SaeFeatureStore`), keyed by `transcript_id` and **upsertable**. Two fill modes:

- **Top-K (default, no re-embed):** the cell below reshapes the `sae_top_features` already in the cache/Baserow into a 16,384-wide sparse matrix. Each protein has its **top-`sae.top_k`** features (default 64) — the *Baserow summary depth is the tunable knob* (`sae.top_k` / `--top-k`). Good for analysis without spending credits.
- **Full pooled vector (opt-in):** set `sae.store_full: true` and re-run the `embed`/`sae` step. That captures **every non-zero** pooled feature (hundreds–thousands/protein) and streams it to the same `.npz` (`data/sae_feature_matrix.npz`), while Baserow keeps the top-K summary. The full per-residue activations aren't recoverable from the cache, so this needs a Biohub re-run.

The cell **won't overwrite** a full store with the top-K reshape — if the store already exists it just loads and reports it. Reload anywhere with `SaeFeatureStore("../data/sae_feature_matrix.npz").to_csr()`.

In [ ]:
from pathlib import Path
from och_annotate.analysis import SaeFeatureStore, update_sae_feature_store

store_path = "../data/sae_feature_matrix.npz"   # == config sae.feature_store_path
if Path(store_path).exists():
    # Don't clobber a store the embed/sae pipeline may have filled with the FULL
    # vector (sae.store_full) — just load and report what's there.
    store = SaeFeatureStore(store_path)
    avg = store.to_csr().nnz / max(len(store), 1)
    kind = "FULL pooled" if avg > 64 + 1 else "top-K"
    print(f"Loaded SAE store ({kind}): {len(store)} proteins x {store.n_features} features, "
          f"avg {avg:.0f} active/protein (model {store.sae_model})")
else:
    # Bootstrap a top-K store from the cache (no re-embed). For the full pooled
    # vector, set sae.store_full=true and re-run the embed/sae step.
    store, stats = update_sae_feature_store(df, store_path)
    print(f"Built top-K SAE store: {len(store)} proteins x {store.n_features} features "
          f"(+{stats['added']} new) -> {store_path}")
    print("  For the FULL pooled vector: set sae.store_full=true and re-run embed/sae.")

## One graph: KNN → UMAP → Leiden (on SAE features)

The neighbor graph is built on the **IDF-weighted** sparse SAE activations (`adata.obsm["X_sae_tfidf"]`), then one UMAP layout and Leiden clusters come off that single graph. Raw activations stay in `adata.X` so the downstream enrichment is computed on untransformed values.

**Parameters were tuned** by a sweep over representation (raw / TF-IDF / TF-IDF→SVD50), `metric`, `n_neighbors`, and `resolution`, scored on a *balanced* objective — agreement with orthogroups (AMI) + neighbor-graph modularity − a tiny-cluster fragmentation penalty (full grid: `data/param_sweep_results.csv`). The winner (**TF-IDF / cosine / `n_neighbors=15` / `resolution=2.0`**) beat the raw `15`/`1.0` baseline on every component, with IDF-weighting the largest lever. `LEIDEN_RES` is the granularity dial; `MIN_DIST` only shapes the UMAP layout, not the clusters.

In [ ]:
import numpy as np
import scanpy as sc
from scipy import sparse
from och_annotate.analysis import build_anndata, sae_enrichment, plot_umap
import plotly.io as pio
pio.renderers.default = "notebook"   # embed interactive plots into nbconvert HTML

# Tuned by a parameter sweep (data/param_sweep_results.csv) on a balanced
# objective: AMI vs orthogroup + graph modularity - tiny-cluster fragmentation.
# Winner beat the raw/cosine/15/1.0 baseline (balanced 2.04 vs 1.05).
SEED        = 0
N_NEIGHBORS = 15
MIN_DIST    = 0.3       # UMAP layout spread only (not used for the graph/clustering)
METRIC      = "cosine"
LEIDEN_RES  = 2.0       # granularity dial -> ~127 clusters; drop to 1.0 (~100) for coarser

adata = build_anndata(df)

# Cluster on IDF-weighted SAE activations: rare, protein-family-specific features
# are upweighted and ubiquitous ones damped (the sweep's biggest lever). Raw
# activations (adata.X) are left untouched so downstream enrichment stays honest.
df_count = np.asarray((adata.X > 0).sum(axis=0)).ravel()
idf = np.log((adata.n_obs + 1) / (df_count + 1)) + 1.0
adata.obsm["X_sae_tfidf"] = adata.X.multiply(sparse.csr_matrix(idf)).tocsr()

sc.pp.neighbors(adata, use_rep="X_sae_tfidf", n_neighbors=N_NEIGHBORS, metric=METRIC, random_state=SEED)
sc.tl.umap(adata, min_dist=MIN_DIST, random_state=SEED)
sc.tl.leiden(adata, resolution=LEIDEN_RES, flavor="igraph", n_iterations=2,
             directed=False, random_state=SEED)

meta_cols = [c for c in df.columns if c not in ("embedding", "sae_top_features")]
coords = df[meta_cols].copy().reset_index(drop=True)
coords["umap_0"] = adata.obsm["X_umap"][:, 0]
coords["umap_1"] = adata.obsm["X_umap"][:, 1]
coords["leiden"] = adata.obs["leiden"].to_numpy()
print(f"{adata.n_obs} proteins; {coords['leiden'].nunique()} Leiden clusters")

In [ ]:
from och_annotate.analysis import plot_umap_searchable

# UMAP colored by chromosome, with a client-side gene search box (works in the
# exported HTML). Searches gene/ortholog/orthogroup ids; matches are ringed.
# embed_js=True here loads plotly.js once for the whole document.
plot_umap_searchable(coords, color="chromosome",
          hover=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup", "chromosome", "leiden"],
          search_fields=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup"],
          labels={"orthogroup": "Orthogroup"},
          title=f"{cfg.name} — UMAP on SAE features (chromosome)", embed_js=True)

In [ ]:
# Same UMAP colored by Leiden cluster, same searchable box. embed_js=False:
# reuse the plotly.js already loaded above (avoids embedding it twice).
plot_umap_searchable(coords, color="leiden",
          hover=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup", "chromosome", "leiden"],
          search_fields=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup"],
          labels={"orthogroup": "Orthogroup"},
          title=f"{cfg.name} — UMAP on SAE features (Leiden clusters)", embed_js=False)

## SAE-feature enrichment per cluster

Wilcoxon rank-sum on the SAE activation matrix, annotated from the **ESM Atlas**: `label`, `category`, `activation_pattern`, `exemplar_protein_families`, top **SwissProt** proteins, and `uniref90_idf` — used to **IDF-weight** markers toward specific (rare) features.

In [ ]:
import pandas as pd
from och_annotate.atlas import fetch_feature_descriptions
from och_annotate.analysis import table_searchable

enrich = sae_enrichment(adata, groupby="leiden", method="wilcoxon", n=15)

# Rich per-feature Atlas metadata for the enriched features (concurrent, cached; no credits)
feat_ids = sorted(enrich["sae_feature"].astype(int).unique())
meta = fetch_feature_descriptions(feat_ids, cache_path="../data/sae_feature_metadata.parquet")
keep = ["feature", "label", "category", "activation_pattern",
        "exemplar_protein_families", "uniref90_idf", "swissprot_top"]
meta = meta[keep].copy(); meta["feature"] = meta["feature"].astype(str)

enrich["feature"] = enrich["sae_feature"].astype(str)
enrich = enrich.merge(meta, on="feature", how="left").drop(columns="feature")

# IDF-weighting: upweight features that are rare across UniRef90 (more specific).
enrich["idf"] = pd.to_numeric(enrich["uniref90_idf"], errors="coerce").fillna(1.0)
enrich["score_idf"] = enrich["scores"] * enrich["idf"]

enrich.to_csv("../data/cluster_sae_enrichment_saebasis.csv", index=False)
print(f"Annotated {len(feat_ids)} features (category / IDF / exemplars / SwissProt); "
      f"{len(enrich)} rows across {enrich['leiden'].nunique()} clusters")

In [ ]:
# Per-cluster functional profile: the category mix of each cluster's top-15 features
profile = (enrich.assign(cluster=enrich["leiden"].astype(int))
                 .groupby("cluster")["category"]
                 .apply(lambda s: ", ".join(f"{c} ({n})" for c, n in
                        s.fillna("(uncat)").replace("", "(uncat)").value_counts().head(4).items()))
                 .rename("top_feature_categories").reset_index())
table_searchable(
    profile, title="Per-cluster functional profile — dominant feature categories",
    caption="Category mix of each cluster's top-15 Wilcoxon markers. "
            "Searchable / paginated (DataTables via CDN; data embedded).",
    page_size=15, max_colwidth=90, order=[["cluster", "asc"]])

In [ ]:
# Top-5 per cluster, ranked by the IDF-WEIGHTED score (specific features rise).
top5 = (enrich.assign(cluster=enrich["leiden"].astype(int))
              .sort_values(["cluster", "score_idf"], ascending=[True, False])
              .groupby("cluster", observed=True).head(5).copy())
top5["rank"] = top5.groupby("cluster").cumcount() + 1
view = top5[["cluster", "rank", "sae_feature", "label", "category", "scores", "idf", "score_idf"]]

# Interactive: search a feature label/category, sort by score_idf, page through clusters.
table_searchable(
    view, title="Cluster SAE-feature enrichment — top 5 per cluster (IDF-weighted)",
    caption="Wilcoxon markers re-ranked by score_idf = score × uniref90_idf. "
            "Searchable / sortable / paginated (DataTables via CDN; data embedded).",
    page_size=15,
    formats={"scores": "{:.1f}", "idf": "{:.2f}", "score_idf": "{:.1f}"},
    max_colwidth=70, order=[["cluster", "asc"], ["score_idf", "desc"]])

In [ ]:
# Rich context for each cluster's lead (IDF-weighted top) feature
lead = top5[top5["rank"] == 1].sort_values("cluster")
for r in lead.itertuples():
    ap = (str(r.activation_pattern) or "").strip().replace("\n", " ")
    ex = (str(r.exemplar_protein_families) or "").strip().splitlines()
    print(f"\u2501\u2501 cluster {r.cluster}  [{r.sae_feature}] {r.label}  ({r.category})")
    print(f"     activation : {ap[:220]}")
    print(f"     exemplars  : {(ex[0][:200] if ex else '')}")
    print(f"     SwissProt  : {r.swissprot_top}")

In [ ]:
# Dotplot of marker SAE features across clusters (Wilcoxon ranking)
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

## Per-cluster feature profile — salience × ubiquity

A second lens on cluster identity, ranked by **mean normalized activation** (what ESMC finds most *salient* across the cluster) with **occurrence** = the fraction of members in which the feature is active. Universal features (occurrence ≈ 100%) define the family; partial ones (≈ 30–60%) flag subgroups or domain variants. This complements the differential Wilcoxon markers above. Reporting the **top 10** per cluster — ranks 6–15 are where subfamily discrimination lives.

> **Residue regions** (start/end/peak per feature) would be the next upgrade, but our cache max-pools SAE activations to one value per protein — positions were discarded. Recovering them needs a per-residue SAE re-run (Biohub credits).

In [ ]:
from och_annotate.analysis import cluster_feature_profile

# Top-10 features per cluster by mean normalized activation (+ occurrence rate)
profile = cluster_feature_profile(adata, groupby="leiden", n=10)

# Atlas labels/category for the profile features (cached; no Biohub credits)
pf_ids = sorted(profile["sae_feature"].unique())
pmeta = fetch_feature_descriptions(pf_ids, cache_path="../data/sae_feature_metadata.parquet")
plabel = dict(zip(pmeta["feature"].astype(int), pmeta["label"]))
pcat = dict(zip(pmeta["feature"].astype(int), pmeta["category"]))
profile["label"] = profile["sae_feature"].map(plabel)
profile["category"] = profile["sae_feature"].map(pcat)
profile.to_csv("../data/cluster_feature_profile_saebasis.csv", index=False)
print(f"Profiled {profile['cluster'].nunique()} clusters x top-10 features "
      f"({len(pf_ids)} unique features)")

In [ ]:
# Grouped top-10 profile per cluster: salience (mean_activation) + ubiquity (occurrence)
pv = (profile.assign(cluster=lambda d: d["cluster"].astype(int))
             .sort_values(["cluster", "rank"])
             [["cluster", "rank", "sae_feature", "label", "category",
               "mean_activation", "occurrence"]])

table_searchable(
    pv, title="Per-cluster feature profile — top 10 by salience × ubiquity",
    caption="mean_activation = salience across cluster members; occurrence = fraction active. "
            "Searchable / sortable / paginated (DataTables via CDN; data embedded).",
    page_size=15,
    formats={"mean_activation": "{:.3f}", "occurrence": "{:.0%}"},
    max_colwidth=70, order=[["cluster", "asc"], ["rank", "asc"]])

## Per-candidate feature report

The per-protein workflow: **top-10 normalized features** with Atlas labels, plus **residue regions** (start–end, peak) for the architecture-bearing top few. The `residues` column is wired but blank until a per-residue SAE run populates it (`sae.residue_regions: true` — same Biohub call, no extra cost; needs a re-run).

In [ ]:
from och_annotate.analysis import candidate_feature_report
from och_annotate.atlas import fetch_all_features

fd = fetch_all_features(cache_path="../data/sae_feature_dictionary.parquet")
full_labels = dict(zip(fd["feature"].astype(int), fd["label"]))

cand = df.iloc[0]   # example candidate; swap in any row / transcript_id
rep = candidate_feature_report(cand["sae_top_features"], labels=full_labels, n=10)
print(f"Candidate {cand.get('transcript_id','?')}  ({cand.get('Ochierchiae_name','')})")
display(rep)
print("residue regions:", "present" if rep["residues"].notna().any()
      else "pending a per-residue SAE run (set sae.residue_regions=true)")

# Sequence novelty relative to the ESM Atlas

How *unusual* is each octopus protein compared to everything the ESM Atlas saw across UniRef90? We answer this directly from the **full pooled SAE vectors** (`data/sae_feature_matrix.npz`, ~1,800 active features/protein — far richer than the top-K summary) combined with the Atlas's **per-feature rarity statistics** (`uniref90_idf`, `uniref90_frequency`). Three complementary signals, each with its own visualization, then a composite index and a ranked candidate list.

1. **IDF-weighted rarity** — does the protein lean on globally-*rare* atlas features? Aggregates each active feature's `uniref90_idf`.
2. **Combinatorial surprise** — how *improbable* is its particular set of active features under the atlas's marginal feature frequencies? Self-information `Σ −log p(feature)`.
3. **Metadata triangulation** — orphan orthogroups / missing mouse orthologs, used as an independent *validation* of whether the SAE-based scores track phylogenetic novelty.

> **Length confound.** The number of active SAE features grows with sequence length, so the raw *sums* of both scores correlate ~0.95–0.999 with `n_active` and are essentially length proxies. We therefore drive the composite off the **per-active-feature means** (and also report rank-within-length-bin versions). Raw sums are kept for reference only.

In [ ]:
# --- Atlas per-feature rarity stats for the UNION of features active anywhere ---
# The full pooled store has all 16,384 features active in >=1 protein. fetch is
# cached (data/sae_feature_metadata.parquet) and free; first run takes a few min.
full = SaeFeatureStore(store_path)
M = full.to_csr().tocsr()                       # proteins x 16,384, float32 pooled activations
nov_ids = full.ids

active_any = np.asarray((M > 0).sum(axis=0)).ravel()
union_feats = sorted(int(j) for j in np.where(active_any > 0)[0])
fmeta = fetch_feature_descriptions(union_feats, cache_path="../data/sae_feature_metadata.parquet")
fmeta["feature"] = fmeta["feature"].astype(int)
flabel = dict(zip(fmeta["feature"], fmeta["label"]))

# Per-feature arrays indexed by SAE feature id. uniref90_idf / uniref90_frequency
# are the ATLAS-WIDE rarity signal; coerce (all atlas cols are strings) and fill
# gaps with the median so a missing stat doesn't masquerade as novelty.
NF = M.shape[1]
idf  = np.full(NF, np.nan); freq = np.full(NF, np.nan)
idf[fmeta["feature"].to_numpy()]  = pd.to_numeric(fmeta["uniref90_idf"], errors="coerce").to_numpy()
freq[fmeta["feature"].to_numpy()] = pd.to_numeric(fmeta["uniref90_frequency"], errors="coerce").to_numpy()
idf  = np.where(np.isnan(idf), np.nanmedian(idf), idf)
freq = np.where(np.isnan(freq) | (freq <= 0), np.nanmedian(freq), freq)

# Combinatorial surprise needs a probability per feature: normalize atlas counts
# to marginals, then self-information -log p (nats). Higher idf => rarer feature.
freq_norm = freq / freq.sum()
surprise_per_feat = -np.log(freq_norm)
print(f"{M.shape[0]} proteins x {NF} features; {len(union_feats)} active anywhere; "
      f"avg {M.nnz / M.shape[0]:.0f} active/protein")

In [ ]:
# --- Per-protein novelty scores over the full SAE vectors (vectorized row sums) ---
act, cols, indptr = M.data.astype("float64"), M.indices, M.indptr
n_active = np.diff(indptr).astype("float64")          # active features per protein (length proxy)

def _seg(vals, reducer=np.add):
    """Per-row reduce of a per-nonzero array, robust to empty rows."""
    out = np.zeros(M.shape[0])
    nz = np.diff(indptr) > 0
    if nz.any():
        out[nz] = reducer.reduceat(vals, indptr[:-1][nz])
    return out

idf_col, surp_col = idf[cols], surprise_per_feat[cols]

# Score 1 — IDF-weighted rarity novelty:
#   (a) raw sum  Sigma(activation * idf)      -- length-confounded
#   (b) weighted-mean idf = sum/Sigma(activation)  -- LENGTH-NORMALIZED (primary)
#   (c) max idf among active features         -- single rarest domain it carries
act_sum   = _seg(act)
idf_sum   = _seg(act * idf_col)
idf_wmean = np.where(act_sum > 0, idf_sum / np.maximum(act_sum, 1e-12), 0.0)
idf_max   = _seg(idf_col, np.maximum)

# Score 3 — Combinatorial surprise: self-information of the active-feature SET
#   raw  Sigma -log p(feature)   (length-confounded)
#   mean per active feature      (LENGTH-NORMALIZED, primary)
surprise_sum  = _seg(surp_col)
surprise_mean = np.where(n_active > 0, surprise_sum / np.maximum(n_active, 1), 0.0)

# Length-normalization cross-check: rank each raw sum WITHIN sequence-length bins
# (20 quantiles of n_active), so a high rank means "rare for a protein of its size".
def _rank_in_bins(x, by, nbins=20):
    b = pd.qcut(pd.Series(by), q=nbins, duplicates="drop")
    return pd.Series(x).groupby(b, observed=True).transform(lambda s: s.rank(pct=True)).to_numpy()

nov = pd.DataFrame({"transcript_id": nov_ids, "n_active": n_active.astype(int)})
nov["idf_sum"], nov["idf_wmean"], nov["idf_max"] = idf_sum, idf_wmean, idf_max
nov["surprise_sum"], nov["surprise_mean"] = surprise_sum, surprise_mean
nov["idf_sum_rankbin"]      = _rank_in_bins(idf_sum, n_active)
nov["surprise_sum_rankbin"] = _rank_in_bins(surprise_sum, n_active)

# Composite novelty index: rank-normalize the two LENGTH-NORMALIZED scores, average.
nov["composite"] = (nov["idf_wmean"].rank(pct=True) + nov["surprise_mean"].rank(pct=True)) / 2.0

print("Length confound (Pearson r with n_active) — sums are length proxies, means are not:")
for c in ["idf_sum", "idf_wmean", "surprise_sum", "surprise_mean"]:
    print(f"  {c:<14}{np.corrcoef(n_active, nov[c])[0, 1]:+.3f}")
print(f"  agreement #1 vs #3 (idf_wmean ~ surprise_mean): "
      f"{np.corrcoef(nov['idf_wmean'], nov['surprise_mean'])[0, 1]:+.3f}")

In [ ]:
# --- Metadata triangulation flags (signal #3) + join names/orthogroups ---
m = df[["transcript_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup"]].copy()
nov = nov.merge(m, on="transcript_id", how="left")

_og = nov["orthogroup"].fillna("").astype(str).str.strip()
_og_counts = _og[_og.str.lower().ne("") & _og.str.lower().ne("nan")].value_counts()
# Orphan/singleton orthogroup: empty/NaN, OR an orthogroup that occurs once in this set.
nov["singleton_og"]  = _og.str.lower().isin(["", "nan"]) | _og.map(lambda o: _og_counts.get(o, 0) == 1)
nov["missing_mouse"] = nov["Mmusculus_gene_name"].fillna("").astype(str).str.strip().str.lower().isin(["", "nan"])

# Per-protein "rarest active features" string (top-3 by atlas idf) for the report table.
def rarest_active(pid, k=3):
    r = M.getrow(nov_ids.index(pid)); ci = r.indices
    if not len(ci):
        return ""
    top = ci[np.argsort(-idf[ci])[:k]]
    return "; ".join(f"[{int(f)}] {flabel.get(int(f), '?')} (idf {idf[int(f)]:.1f})" for f in top)

print(f"flags: {nov['missing_mouse'].mean():.1%} missing mouse ortholog, "
      f"{nov['singleton_og'].mean():.1%} singleton/orphan orthogroup (baseline rates)")
nov[["transcript_id", "n_active", "idf_wmean", "surprise_mean", "composite",
     "singleton_og", "missing_mouse"]].describe(include="all").loc[["mean", "min", "max"]]

### Score distributions

One panel per signal: the length-normalized IDF-weighted rarity (#1), the length-normalized combinatorial surprise (#3), and the composite index. Long right tails are the candidates of interest.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
panels = [
    ("idf_wmean",     "#1 IDF-weighted rarity\n(mean idf / active feature)", "#3b6fb6"),
    ("surprise_mean", "#3 Combinatorial surprise\n(mean −log p / active feature)", "#b6603b"),
    ("composite",     "Composite novelty index\n(rank-avg of #1 & #3)", "#4a7a4a"),
]
for ax, (col, ttl, c) in zip(axes, panels):
    ax.hist(nov[col], bins=60, color=c, alpha=0.85)
    ax.axvline(nov[col].quantile(0.9), color="#222", ls="--", lw=1, label="90th pct")
    ax.set_title(ttl, fontsize=10); ax.set_xlabel(col); ax.set_ylabel("proteins")
    ax.legend(fontsize=8, frameon=False)
fig.suptitle(f"{cfg.name} — novelty score distributions (n={len(nov):,})", fontsize=12, y=1.04)
fig.tight_layout(); plt.show()

### Novelty on the UMAP, and do the two signals agree?

The same SAE-feature UMAP from above, now colored by the **composite novelty index** — novel proteins should pool in specific regions rather than scatter at random. The searchable box still finds genes by id. Below it, a scatter of signal **#1 vs #3** (colored by the missing-mouse-ortholog flag) shows how tightly the two independent rarity measures track each other.

In [ ]:
# UMAP colored by the composite novelty index (continuous color). Join scores
# onto the clustering `coords` by transcript_id; embed_js=False reuses plotly.js.
coords_nov = coords.merge(
    nov[["transcript_id", "idf_wmean", "surprise_mean", "composite", "n_active",
         "singleton_og", "missing_mouse"]],
    on="transcript_id", how="left",
)
plot_umap_searchable(
    coords_nov, color="composite",
    hover=["transcript_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup",
           "leiden", "composite", "idf_wmean", "surprise_mean", "n_active"],
    search_fields=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup"],
    labels={"orthogroup": "Orthogroup", "composite": "Composite novelty"},
    title=f"{cfg.name} — UMAP on SAE features (composite novelty)", embed_js=False)

In [ ]:
# Agreement between the two independent signals (#1 vs #3), colored by the
# missing-mouse-ortholog flag. A point sample keeps the static HTML light.
samp = nov.sample(min(8000, len(nov)), random_state=SEED)
r = np.corrcoef(nov["idf_wmean"], nov["surprise_mean"])[0, 1]
fig, ax = plt.subplots(figsize=(5.6, 5.2))
for flag, c, lab in [(False, "#3b6fb6", "has mouse ortholog"), (True, "#d1495b", "missing mouse ortholog")]:
    s = samp[samp["missing_mouse"] == flag]
    ax.scatter(s["idf_wmean"], s["surprise_mean"], s=6, alpha=0.35, c=c, label=lab, linewidths=0)
ax.set_xlabel("#1 IDF-weighted rarity (mean idf)")
ax.set_ylabel("#3 combinatorial surprise (mean −log p)")
ax.set_title(f"Signal agreement (Pearson r = {r:.2f}, n={len(samp):,} sampled)")
ax.legend(fontsize=8, frameon=False, markerscale=2)
fig.tight_layout(); plt.show()

### Validation (#3): do SAE-novel proteins enrich for orphan metadata?

Cephalopods carry many lineage-specific genes, so *if* the SAE rarity scores were tracking phylogenetic novelty we'd expect the top-novelty decile to be **enriched** for missing-mouse-ortholog and singleton-orthogroup flags relative to the whole proteome. We test each score's top decile against the baseline rates.

In [ ]:
# Top-decile flag rates vs baseline, for each score and the composite.
rows = []
base_mouse, base_og = nov["missing_mouse"].mean(), nov["singleton_og"].mean()
for sc in ["idf_wmean", "surprise_mean", "composite"]:
    top = nov[nov[sc] >= nov[sc].quantile(0.9)]
    rows.append({
        "score": sc, "top_decile_n": len(top),
        "missing_mouse_top": top["missing_mouse"].mean(), "missing_mouse_base": base_mouse,
        "missing_mouse_enrich": top["missing_mouse"].mean() / base_mouse,
        "singleton_og_top": top["singleton_og"].mean(), "singleton_og_base": base_og,
        "singleton_og_enrich": top["singleton_og"].mean() / base_og,
    })
valid = pd.DataFrame(rows)
print("Enrichment ratio < 1 means the SAE-novel top decile is DEPLETED of that flag.")
valid.style.format({c: "{:.3f}" for c in valid.columns if c != "score" and c != "top_decile_n"})

### Top novel candidates

The highest-composite proteins, with both signals, the metadata flags, and each protein's three **rarest active SAE features** (id + Atlas label + idf). Searchable / sortable / paginated — click a column header to re-sort (e.g. by `surprise_mean` alone).

In [ ]:
from och_annotate.analysis import table_searchable

TOP_N = 100
cand = nov.sort_values("composite", ascending=False).head(TOP_N).copy()
cand["rarest_active_features"] = cand["transcript_id"].map(lambda p: rarest_active(p, k=3))
cand_view = cand[[
    "transcript_id", "Ochierchiae_name", "Mmusculus_gene_name", "orthogroup",
    "singleton_og", "missing_mouse", "n_active",
    "idf_wmean", "surprise_mean", "composite", "rarest_active_features",
]].rename(columns={
    "Ochierchiae_name": "octopus_name", "Mmusculus_gene_name": "mouse_ortholog",
    "idf_wmean": "idf_rarity", "surprise_mean": "surprise",
})
nov.sort_values("composite", ascending=False).to_csv("../data/novelty_scores_saebasis.csv", index=False)

table_searchable(
    cand_view,
    title=f"Top {TOP_N} novel candidates (by composite index)",
    caption="Searchable / sortable / paginated (DataTables via CDN — needs internet for interactivity; "
            "data is embedded). Saved full ranking to data/novelty_scores_saebasis.csv.",
    page_size=15,
    formats={"idf_rarity": "{:.3f}", "surprise": "{:.3f}", "composite": "{:.4f}",
             "singleton_og": lambda v: "yes" if v else "", "missing_mouse": lambda v: "yes" if v else ""},
    max_colwidth=70,
    order=[["composite", "desc"]],
)

### Notes on the Atlas annotations

All metadata comes from the public **ESM Atlas** feature API
(`biohub.ai/esm/protein/api/v1alpha1/features/{idx}`) via
`och_annotate.atlas.fetch_feature_descriptions` — cached under `data/`, **not** charged
against Biohub embedding credits. The enrichment table is ranked by
`score_idf = wilcoxon_score × uniref90_idf` so cluster-specific (rare) features rise above
ubiquitous ones; the lead-feature block adds activation pattern, exemplar families and
reviewed-UniProt examples. The novelty section reuses the same Atlas rarity stats
(`uniref90_idf`, `uniref90_frequency`) over the full pooled SAE vectors.

**Interactive tables.** The large tables (cluster enrichment, per-cluster profile, novelty
candidates) are rendered with `analysis.table_searchable` — searchable, sortable and paginated
via **DataTables.js + jQuery from a CDN**. The row *data* is embedded in the HTML, but unlike
the inline plotly figures the table *interactivity* needs an internet connection to load the
CDN scripts; offline they fall back to a plain static table.

**Full outputs:** `data/cluster_sae_enrichment_saebasis.csv`,
`data/cluster_feature_profile_saebasis.csv`, `data/novelty_scores_saebasis.csv`.

### Other next steps
- Write `adata.obs["leiden"]` and the composite novelty back to Baserow.
- GO-enrich each cluster from the SwissProt example proteins.
- Tune `LEIDEN_RES`, `N_NEIGHBORS`, `MIN_DIST` (the params cell above).